# Bank Customer Churn Prediction

 Task Overview

Our goal is to build a predictive model to identify customers who are likely to churn (stop using the bank's services). This will help the bank focus retention efforts on high-risk customers.

We will cover:
* Data preprocessing techniques.
* Building and evaluating a machine learning model for classification.
* Interpreting model results.

For this exercise, we will assume a customer churn dataset. Since no specific file path was provided for the churn data, I will create a synthetic dataset for demonstration purposes. In a real-world scenario, you would replace this with your actual data loading step.

In [46]:
import pandas as pd
import numpy as np

# #TODO: Load your actual customer churn dataset here.
# # For demonstration, we'll create a synthetic dataset.
# try:
#     df = pd.read_csv('your_churn_dataset.csv')
# except FileNotFoundError:
#     print('Churn dataset not found. Generating synthetic data...')

# Generate synthetic data for demonstration
np.random.seed(42)
num_customers = 1000

data = {
    'CustomerId': range(1, num_customers + 1),
    'CreditScore': np.random.randint(350, 850, num_customers),
    'Geography': np.random.choice(['France', 'Spain', 'Germany'], num_customers),
    'Gender': np.random.choice(['Male', 'Female'], num_customers),
    'Age': np.random.randint(18, 92, num_customers),
    'Tenure': np.random.randint(0, 10, num_customers),
    'Balance': np.random.uniform(0, 250000, num_customers),
    'NumOfProducts': np.random.randint(1, 4, num_customers),
    'HasCrCard': np.random.choice([0, 1], num_customers),
    'IsActiveMember': np.random.choice([0, 1], num_customers),
    'EstimatedSalary': np.random.uniform(0, 200000, num_customers),
    'Exited': np.random.choice([0, 1], num_customers, p=[0.8, 0.2]) # 20% churn rate
}

df = pd.DataFrame(data)

print('Dataset loaded/generated successfully. Here are the first 5 rows:')
display(df.head())

Dataset loaded/generated successfully. Here are the first 5 rows:


,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,452,France,Male,82,6,162079.501863,3,0,0,185012.911289,0
1,2,785,Germany,Male,84,1,18526.355644,3,0,1,73222.623586,0
2,3,698,France,Male,69,5,93867.232117,3,1,1,20727.345296,0
3,4,620,Spain,Female,79,5,200953.636298,3,0,1,140481.073173,0
4,5,456,Germany,Male,40,2,108368.058189,3,1,0,59891.544787,1


  Initial Data Exploration

Let's get a basic understanding of our dataset's structure, data types, and check for any missing values.

In [47]:
print('DataFrame Info:')
df.info()

print('\nDescriptive Statistics:')
display(df.describe())

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerId       1000 non-null   int64  
 1   CreditScore      1000 non-null   int64  
 2   Geography        1000 non-null   object 
 3   Gender           1000 non-null   object 
 4   Age              1000 non-null   int64  
 5   Tenure           1000 non-null   int64  
 6   Balance          1000 non-null   float64
 7   NumOfProducts    1000 non-null   int64  
 8   HasCrCard        1000 non-null   int64  
 9   IsActiveMember   1000 non-null   int64  
 10  EstimatedSalary  1000 non-null   float64
 11  Exited           1000 non-null   int64  
dtypes: float64(2), int64(8), object(2)
memory usage: 93.9+ KB

Descriptive Statistics:


,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,500.500000,600.68300,53.809000,4.487000,123309.241461,2.012000,0.531000,0.497000,100129.635250,0.219000
std,288.819436,141.79995,21.029997,2.901077,71421.873194,0.818857,0.499288,0.500241,56886.578943,0.413776
min,1.000000,350.00000,18.000000,0.000000,7.679711,1.000000,0.000000,0.000000,125.503326,0.000000
25%,250.750000,483.75000,36.000000,2.000000,62105.301508,1.000000,0.000000,0.000000,52470.326128,0.000000
50%,500.500000,596.00000,53.500000,5.000000,122377.124932,2.000000,1.000000,0.000000,102292.970044,0.000000
75%,750.250000,723.00000,72.000000,7.000000,183626.429804,3.000000,1.000000,1.000000,146965.579159,0.000000
max,1000.000000,849.00000,91.000000,9.000000,249889.425813,3.000000,1.000000,1.000000,199969.848897,1.000000


oui

In [51]:

# PRÉDICTION DU TAUX DE DÉSABONNEMENT BANCAIRE
# Machine Learning avec Scikit-learn


import pandas as pd
import numpy as np
# Removed: import matplotlib.use('Agg') to allow inline plotting
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, roc_auc_score, roc_curve,
                              ConfusionMatrixDisplay)
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Style global
plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.edgecolor'] = '#444'
plt.rcParams['grid.color'] = '#2a2d3a'
plt.rcParams['grid.alpha'] = 0.5

ACCENT   = '#6366f1'   # indigo
GREEN    = '#10b981'
RED      = '#ef4444'
ORANGE   = '#f59e0b'
CYAN     = '#06b6d4'
PALETTE  = [ACCENT, GREEN, RED, ORANGE, CYAN, '#8b5cf6', '#ec4899']

print("=" * 60)
print("  ÉTAPE 1 : CHARGEMENT ET EXPLORATION DES DONNÉES")
print("=" * 60)

# The dataframe 'df' was already created in a previous cell (synthetic data).
# Original line: df = pd.read_csv('/home/claude/Churn_Modelling.csv')
# Ensure 'df' from the previous cell is used.

print(f"\n Dataset généré : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"\n Aperçu des colonnes :\n{df.dtypes}")
print(f"\n Statistiques descriptives :\n{df.describe().round(2)}")
print(f"\n Valeurs manquantes :\n{df.isnull().sum()[df.isnull().sum()>0]}")
print(f"\n Distribution du churn :\n{df['Exited'].value_counts()}")
print(f"   Taux de churn : {df['Exited'].mean():.2%}")


print("\n" + "=" * 60)
print("  ÉTAPE 2 : VISUALISATION EXPLORATOIRE (EDA)")
print("=" * 60)

fig = plt.figure(figsize=(20, 16), facecolor='#0f1117')
fig.suptitle('Analyse Exploratoire — Churn Bancaire',
             fontsize=22, fontweight='bold', color='white', y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

# 1. Taux de churn global
ax1 = fig.add_subplot(gs[0, 0])
vals = df['Exited'].value_counts()
wedges, texts, autotexts = ax1.pie(
    vals, labels=['Fidèle', 'Churné'],
    colors=[GREEN, RED], autopct='%1.1f%%',
    startangle=90, textprops={'color':'white','fontsize':11},
    wedgeprops={'edgecolor':'#0f1117','linewidth':2})
for at in autotexts: at.set_fontweight('bold')
ax1.set_title('Distribution du Churn', fontweight='bold', color='white', pad=12)

# 2. Churn par géographie
ax2 = fig.add_subplot(gs[0, 1])
geo = df.groupby('Geography')['Exited'].mean().sort_values(ascending=False) * 100
bars = ax2.bar(geo.index, geo.values, color=[RED, ORANGE, GREEN], edgecolor='none', width=0.55)
for b in bars:
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
             f'{b.get_height():.1f}%', ha='center', va='bottom',
             color='white', fontsize=10, fontweight='bold')
ax2.set_title('Taux de Churn par Pays', fontweight='bold', color='white')
ax2.set_ylabel('Taux de churn (%)', color='white')
ax2.set_ylim(0, geo.max()*1.2)

# 3. Churn par genre
ax3 = fig.add_subplot(gs[0, 2])
gen = df.groupby('Gender')['Exited'].mean() * 100
ax3.bar(gen.index, gen.values, color=[ACCENT, CYAN], edgecolor='none', width=0.45)
for i, (idx, v) in enumerate(gen.items()):
    ax3.text(i, v+0.2, f'{v:.1f}%', ha='center', va='bottom',
             color='white', fontsize=11, fontweight='bold')
ax3.set_title('Taux de Churn par Genre', fontweight='bold', color='white')
ax3.set_ylabel('Taux de churn (%)', color='white')
ax3.set_ylim(0, gen.max()*1.25)

# 4. Distribution de l'âge selon churn
ax4 = fig.add_subplot(gs[1, 0])
df[df['Exited']==0]['Age'].plot.hist(bins=30, alpha=0.65, color=GREEN, ax=ax4, label='Fidèle')
df[df['Exited']==1]['Age'].plot.hist(bins=30, alpha=0.65, color=RED, ax=ax4, label='Churné')
ax4.set_title('Distribution Âge vs Churn', fontweight='bold', color='white')
ax4.set_xlabel('Âge', color='white')
ax4.legend(facecolor='#1a1d27', labelcolor='white')

# 5. Balance vs Churn (boxplot)
ax5 = fig.add_subplot(gs[1, 1])
data_box = [df[df['Exited']==0]['Balance'].dropna(),
            df[df['Exited']==1]['Balance'].dropna()]
bp = ax5.boxplot(data_box, patch_artist=True,
                  medianprops={'color':'white','linewidth':2})
bp['boxes'][0].set_facecolor(GREEN)
bp['boxes'][1].set_facecolor(RED)
for whisker in bp['whiskers']: whisker.set_color('#aaa')
for cap in bp['caps']: cap.set_color('#aaa')
ax5.set_xticklabels(['Fidèle', 'Churné'], color='white')
ax5.set_title('Solde du Compte vs Churn', fontweight='bold', color='white')
ax5.set_ylabel('Balance (€)', color='white')

# 6. Nombre de produits vs Churn
ax6 = fig.add_subplot(gs[1, 2])
prod = df.groupby('NumOfProducts')['Exited'].mean() * 100
ax6.bar(prod.index.astype(str), prod.values,
        color=[GREEN, ORANGE, RED, '#9333ea'], edgecolor='none', width=0.55)
for i, v in enumerate(prod.values):
    ax6.text(i, v+0.3, f'{v:.1f}%', ha='center', va='bottom',
             color='white', fontsize=10, fontweight='bold')
ax6.set_title('Churn par Nbre de Produits', fontweight='bold', color='white')
ax6.set_xlabel('Nombre de produits', color='white')
ax6.set_ylabel('Taux de churn (%)', color='white')

# 7. Heatmap de corrélation
ax7 = fig.add_subplot(gs[2, :])
num_cols = ['CreditScore','Age','Tenure','Balance','NumOfProducts',
            'HasCrCard','IsActiveMember','EstimatedSalary','Exited']
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 10, as_cmap=True)
sns.heatmap(corr, mask=mask, ax=ax7, annot=True, fmt='.2f',
            cmap=cmap, center=0, linewidths=0.5, linecolor='#0f1117',
            cbar_kws={'shrink':0.6}, annot_kws={'size':9})
ax7.set_title('Matrice de Corrélation', fontweight='bold', color='white', pad=12)
ax7.tick_params(colors='white', labelsize=9)

plt.savefig('./fig1_eda.png', dpi=140, bbox_inches='tight',
            facecolor='#0f1117')
# Removed: plt.close() to display plots inline
print(" Figure EDA sauvegardée")

print("\n" + "=" * 60)
print("  ÉTAPE 3 : PRÉTRAITEMENT DES DONNÉES")
print("=" * 60)

# Suppression des colonnes non pertinentes
df_clean = df.drop(columns=['CustomerId'])
# Original line: df_clean = df.drop(columns=['RowNumber','CustomerId','Surname'])
# 'RowNumber' and 'Surname' were not present in the synthetic dataset, so removed from drop list.
print(f"Colonnes supprimées : CustomerId")

# Gestion des valeurs manquantes
num_imputer = SimpleImputer(strategy='median')
for col in ['CreditScore','Age','Balance']:
    df_clean[col] = num_imputer.fit_transform(df_clean[[col]])
print(f" Valeurs manquantes imputées par la médiane")

# Encodage des variables catégorielles
le_gender = LabelEncoder()
df_clean['Gender'] = le_gender.fit_transform(df_clean['Gender'])

geo_dummies = pd.get_dummies(df_clean['Geography'], prefix='Geo', drop_first=True)
df_clean = pd.concat([df_clean.drop('Geography', axis=1), geo_dummies], axis=1)
print(f" Encodage Label (Gender) + One-Hot (Geography)")
print(f"Shape après prétraitement : {df_clean.shape}")

# Séparation features / target
X = df_clean.drop('Exited', axis=1)
y = df_clean['Exited']
print(f"\n Features ({X.shape[1]}) : {list(X.columns)}")
print(f"   Target : Exited (0/1)")

# Train/Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"\n Train : {X_train.shape[0]:,} | Test : {X_test.shape[0]:,}")
print(f"   Churn train : {y_train.mean():.2%} | Churn test : {y_test.mean():.2%}")

# Normalisation (IMPORTANT : fit sur train seulement !)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)   # ← transform seulement, pas fit_transform
print(" Normalisation StandardScaler (fit sur train uniquement)")


print("\n" + "=" * 60)
print("  ÉTAPE 4 : CONSTRUCTION ET ENTRAÎNEMENT DES MODÈLES")
print("=" * 60)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred  = model.predict(X_test_sc)
    y_proba = model.predict_proba(X_test_sc)[:,1]

    acc     = accuracy_score(y_test, y_pred)
    auc     = roc_auc_score(y_test, y_proba)
    report  = classification_report(y_test, y_pred, output_dict=True)

    results[name] = {'model': model, 'y_pred': y_pred, 'y_proba': y_proba,
                     'accuracy': acc, 'auc': auc, 'report': report}
    print(f"\n{'─'*50}")
    print(f"  {name}")
    print(f"{'─'*50}")
    print(f"  Accuracy : {acc:.4f} ({acc*100:.2f}%)")
    print(f"  AUC-ROC  : {auc:.4f}")
    print(f"  Rapport détaillé :\n{classification_report(y_test, y_pred, target_names=['Fidèle','Churné'])}")


print("\n" + "=" * 60)
print("  ÉTAPE 5 : VISUALISATION DES PERFORMANCES")
print("=" * 60)

fig2, axes = plt.subplots(2, 3, figsize=(20, 12), facecolor='#0f1117')
fig2.suptitle('Évaluation des Modèles — Prédiction du Churn',
              fontsize=20, fontweight='bold', color='white', y=0.98)

colors_model = [ACCENT, ORANGE, GREEN]

# Matrices de confusion
for i, (name, res) in enumerate(results.items()):
    ax = axes[0, i]
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fidèle','Churné'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix\n{name}', fontweight='bold', color='white', fontsize=11)
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    ax.tick_params(colors='white')
    for text in ax.texts: text.set_color('white')
    ax.set_facecolor('#1a1d27')
    ax.images[0].set_cmap(plt.cm.get_cmap('Blues'))

# Courbes ROC
ax_roc = axes[1, 0]
for (name, res), col in zip(results.items(), colors_model):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax_roc.plot(fpr, tpr, color=col, lw=2.5,
                label=f"{name} (AUC={res['auc']:.3f})")
ax_roc.plot([0,1],[0,1],'--', color='#666', lw=1.5, label='Aléatoire')
ax_roc.set_xlabel('Taux de faux positifs', color='white')
ax_roc.set_ylabel('Taux de vrais positifs', color='white')
ax_roc.set_title('Courbes ROC', fontweight='bold', color='white')
ax_roc.legend(facecolor='#1a1d27', labelcolor='white', fontsize=9)
ax_roc.set_facecolor('#1a1d27')
ax_roc.grid(True, alpha=0.3)

# Comparaison des métriques (barres groupées)
ax_met = axes[1, 1]
metric_names = ['Accuracy','AUC-ROC','Precision (churné)','Recall (churné)']
x = np.arange(len(metric_names))
width = 0.25
for i, (name, res) in enumerate(results.items()):
    vals = [
        res['accuracy'],
        res['auc'],
        res['report']['1']['precision'],
        res['report']['1']['recall']
    ]
    bars = ax_met.bar(x + i*width, vals, width, label=name,
                      color=colors_model[i], alpha=0.85, edgecolor='none')
ax_met.set_xticks(x + width)
ax_met.set_xticklabels(metric_names, rotation=15, ha='right', fontsize=9, color='white')
ax_met.set_ylim(0, 1.1)
ax_met.set_title('Comparaison des Métriques', fontweight='bold', color='white')
ax_met.legend(facecolor='#1a1d27', labelcolor='white', fontsize=9)
ax_met.set_facecolor('#1a1d27')
ax_met.grid(axis='y', alpha=0.3)
ax_met.axhline(1.0, color='#555', linestyle='--', lw=1)

# Importance des features (Random Forest)
ax_feat = axes[1, 2]
rf_model = results['Random Forest']['model']
importances = rf_model.feature_importances_
feat_df = pd.DataFrame({'feature': X.columns, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=True).tail(10)
colors_feat = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(feat_df)))
ax_feat.barh(feat_df['feature'], feat_df['importance'],
             color=colors_feat, edgecolor='none')
ax_feat.set_title('Importance des Variables (RF)', fontweight='bold', color='white')
ax_feat.set_xlabel('Importance', color='white')
ax_feat.set_facecolor('#1a1d27')
ax_feat.grid(axis='x', alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('./fig2_models.png', dpi=140, bbox_inches='tight',
            facecolor='#0f1117')
# Removed: plt.close() to display plots inline
print(" Figure évaluation des modèles sauvegardée")


print("\n" + "=" * 60)
print("  ÉTAPE 6 : ANALYSE DES CLIENTS À RISQUE")
print("=" * 60)

# Utilisation du meilleur modèle (Random Forest)
best_model = results['Random Forest']['model']
X_test_original = X_test.copy()
X_test_original['Churn_Probability'] = results['Random Forest']['y_proba']
X_test_original['Churn_Predicted']   = results['Random Forest']['y_pred']
X_test_original['Churn_Actual']      = y_test.values

high_risk = X_test_original[X_test_original['Churn_Probability'] > 0.7]
print(f"\n  Clients à HAUT RISQUE (prob > 70%) : {len(high_risk):,}")
print(f"   Soit {len(high_risk)/len(X_test_original)*100:.1f}% du portefeuille test")
print(f"\n   Profil moyen des clients à haut risque :")
print(high_risk[['Age','Balance','NumOfProducts','IsActiveMember','Churn_Probability']].describe().round(2))

print("\n" + "=" * 60)
print("  SYNTHÈSE FINALE DES PERFORMANCES")
print("=" * 60)
print(f"\n{'Modèle':<22} {'Accuracy':>10} {'AUC-ROC':>10} {'Precision':>10} {'Recall':>10}")
print("─" * 65)
for name, res in results.items():
    print(f"{name:<22} {res['accuracy']:>10.4f} {res['auc']:>10.4f} "
          f"{res['report']['1']['precision']:>10.4f} "
          f"{res['report']['1']['recall']:>10.4f}")

best_name = max(results, key=lambda k: results[k]['auc'])
print(f"\n Meilleur modèle : {best_name} (AUC = {results[best_name]['auc']:.4f})")
print("\n Analyse complète terminée !")


  ÉTAPE 1 : CHARGEMENT ET EXPLORATION DES DONNÉES

 Dataset généré : 1,000 lignes × 12 colonnes

 Aperçu des colonnes :
CustomerId           int64
CreditScore          int64
Geography           object
Gender              object
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object

 Statistiques descriptives :
       CustomerId  CreditScore      Age   Tenure    Balance  NumOfProducts  \
count     1000.00      1000.00  1000.00  1000.00    1000.00        1000.00   
mean       500.50       600.68    53.81     4.49  123309.24           2.01   
std        288.82       141.80    21.03     2.90   71421.87           0.82   
min          1.00       350.00    18.00     0.00       7.68           1.00   
25%        250.75       483.75    36.00     2.00   62105.30           1.00   
50%        500.50       596.00    53.50     